## Lambda and API Gateway Basics

### Introduction: Serverless and AWS Lambda

Welcome to the first lesson of the "Building Serverless Applications" course. In this lesson, we will explore the basics of serverless computing and how AWS Lambda helps you build applications without managing servers.

Serverless computing means you do not have to worry about setting up or maintaining servers. Instead, you write your code, and the cloud provider (like AWS) runs it for you only when needed. This makes it easier to scale and can save money, since you only pay for what you use.

AWS Lambda is a popular serverless service. With Lambda, you write small pieces of code called functions. These functions run in response to events, such as a user making a request to your application.

In this lesson, you will learn how to connect AWS Lambda to an API Gateway so you can build an API that responds to user requests. By the end, you will understand how a simple API can trigger a Lambda function, process some data, and return a result.

---

## Recall: APIs and API Gateway

Before we dive in, let's quickly remind ourselves what an API (Application Programming Interface) is and how API Gateway fits in.

An API (Application Programming Interface) is a way for different programs to talk to each other. For example, when you use a weather app, it might call an API to get the latest weather data.

API Gateway is a service from AWS that helps you create and manage APIs. It acts as a "front door" for your application. When someone sends a request to your API, API Gateway receives it and can pass it on to a Lambda function to process.

In this lesson, we will see how API Gateway and Lambda work together to handle a simple calculation request.

---

## Exploring the Lambda Function Code

Let's build up the Lambda function step by step. This function will receive a request with an amount, calculate tax, and return the total.

### Step 1: Importing Required Libraries

First, we need to import the `json` library. This helps us work with JSON data, which is a common format for sending information in APIs.

```python
import json
```

### Step 2: Defining the Lambda Handler

Every Lambda function needs a handler. The handler is the main function that AWS Lambda calls when the function is triggered.

```python
def handler(event, context):
```

* `event` contains information about the request, such as the data sent by the user.
* `context` contains information about the runtime and the function itself. For now, we won't use it, but it must be included.

### Step 3: Reading the Input Data

We want to get the amount from the request. The data is sent as a JSON string in the body of the event.

```python
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
```

* `event.get("body")` gets the body of the request.
* `json.loads(...)` converts the JSON string into a Python dictionary.
* `body.get("amount", 0)` gets the amount value from the dictionary. If it's not there, it uses 0.
* We use `float(...)` to make sure the amount is a number.

### Step 4: Calculating Tax and Total

Now, let's calculate the tax and the total amount.

```python
    tax = round(amount * 0.07, 2)
    total = round(amount + tax, 2)
```

* We multiply the amount by 0.07 to get a 7% tax.
* `round(..., 2)` makes sure the result has two decimal places, like money.

### Step 5: Returning the Response

Finally, we need to return the result as a JSON response.

```python
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"amount": amount, "tax": tax, "total": total})
    }
```

* `statusCode: 200` means the request was successful.
* `headers` tells the client that the response is in JSON format.
* `body` contains the result as a JSON string.

**Full Example Output**

If you send a request with an amount of 100, the output will look like this:

```text
{
  "amount": 100.0,
  "tax": 7.0,
  "total": 107.0
}
```

---

## Understanding the SAM Template

To connect API Gateway to our Lambda function, we use a template file called `template.yaml`. This file uses AWS SAM (Serverless Application Model) to define our resources.

Let's look at the key parts:

```yaml
Resources:
  CalcFunction:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: .
      Handler: main.handler
      Runtime: python3.13
      Timeout: 5
      MemorySize: 256
      Events:
        Api:
          Type: HttpApi
          Properties:
            Path: /calculate
            Method: POST
```

* `Type: AWS::Serverless::Function` tells AWS this is a Lambda function.
* `Handler: main.handler` means the function to run is called `handler` in the `main.py` file.
* `Runtime: python3.13` sets the Python version.
* `Events` defines how the function is triggered. Here, it's triggered by an HTTP API at the path `/calculate` using the `POST` method.

This template tells AWS to create a Lambda function and set up API Gateway so that when someone sends a POST request to `/calculate`, our function runs. SAM automatically creates the necessary IAM execution role for the Lambda function with the basic permissions it needs to run and write logs to CloudWatch.

---

## Building and Deploying with SAM

Now that we understand the Lambda function and SAM template, let's learn how to actually build and deploy our serverless application.

AWS SAM provides two key commands to get your application running.

### Building the Application

First, we need to build our application:

```shell
sam build
```

This command:

* Reads the `template.yaml` file
* Packages your Lambda function code
* Downloads any dependencies
* Prepares everything for deployment

You'll see output showing SAM building your function and creating a `.aws-sam` folder with the packaged code.

**Example Output:**

```text
Starting Build use cache
Cache is invalid, running fresh build
Building codeuri: . runtime: python3.13 metadata: {} architecture: x86_64 functions: CalcFunction
Running PythonPipBuilder:CopySource
Running PythonPipBuilder:CopySource

Build Succeeded

Built Artifacts  : .aws-sam/build
Built Template   : .aws-sam/build/template.yaml

Commands you can use next
=========================
[*] Validate SAM template: sam validate
[*] Invoke Function: sam local invoke
[*] Test Function in the cloud: sam sync --stack-name {{stack-name}} --watch
[*] Deploy: sam deploy --guided
```

### Deploying the Application

Next, we deploy the application to AWS:

```shell
sam deploy --guided
```

The `--guided` flag walks you through the deployment process step by step:

* It asks for a stack name (like `tax-calculator-app`)
* Confirms the AWS region to deploy to
* Shows you what resources will be created
* Asks for confirmation before deploying

**Example Output:**

```text
Configuring SAM deploy
======================

    Looking for config file [samconfig.toml] :  Not found

    Setting default arguments for 'sam deploy'
    =========================================
    Stack Name [sam-app]: tax-calculator-app
    AWS Region [us-east-1]: 
    #Shows you resources changes to be deployed and require a 'Y' to initiate deploy
    Confirm changes before deploy [y/N]: y
    #SAM needs permission to be able to create roles to connect to the resources in your template
    Allow SAM CLI to create IAM roles [Y/n]: Y
    #Preserves the state of previously provisioned resources when an operation fails
    Disable rollback [y/N]: N
    CalcFunction has no authentication. Is this okay? [y/N]: y
    Save parameters to config file [Y/n]: Y
    SAM configuration file [samconfig.toml]: 
    SAM configuration environment [default]: 

    Deploy this changeset? [y/N]: y

2024-01-15 10:30:15 - Waiting for stack create/update to complete

CloudFormation events from stack operations (most recent events last)
---------------------------------------------------------------------
2024-01-15 10:30:18	tax-calculator-app	CREATE_IN_PROGRESS	User Initiated
2024-01-15 10:30:21	CalcFunctionRole	CREATE_IN_PROGRESS	
2024-01-15 10:30:22	CalcFunctionRole	CREATE_IN_PROGRESS	Resource creation Initiated
2024-01-15 10:30:35	CalcFunctionRole	CREATE_COMPLETE	
2024-01-15 10:30:37	CalcFunction	CREATE_IN_PROGRESS	
2024-01-15 10:30:38	CalcFunction	CREATE_IN_PROGRESS	Resource creation Initiated
2024-01-15 10:30:48	CalcFunction	CREATE_COMPLETE	
2024-01-15 10:30:50	ServerlessHttpApi	CREATE_IN_PROGRESS	
2024-01-15 10:30:51	ServerlessHttpApi	CREATE_IN_PROGRESS	Resource creation Initiated
2024-01-15 10:30:53	ServerlessHttpApi	CREATE_COMPLETE	
2024-01-15 10:30:54	tax-calculator-app	CREATE_COMPLETE	

Successfully created/updated stack - tax-calculator-app in us-east-1

Outputs
---------
Key                 CalcApi
Description         API Gateway endpoint URL for Calc function
Value               https://abc123def4.execute-api.us-east-1.amazonaws.com/calculate
```

After deployment completes, SAM will show you the API Gateway endpoint URL where you can send requests to your Lambda function.

**Example Workflow**

Here's what a typical development cycle looks like:

* Write your Lambda function code (`main.py`)
* Define your resources in `template.yaml`
* Run `sam build` to package everything
* Run `sam deploy --guided` to deploy to AWS
* Test your API using the provided endpoint URL

For subsequent deployments, you can simply run `sam deploy` without the `--guided` flag since SAM remembers your configuration.

---

## Putting It All Together

Let's see how everything works as a whole:

* A user sends a POST request to `/calculate` with a JSON body like `{"amount": 100}`.
* API Gateway receives the request and passes it to the Lambda function.
* The Lambda function reads the amount, calculates the tax and total, and returns the result.
* API Gateway sends the response back to the user.

**Real-World Example**

Imagine you are building a simple online store. When a customer enters the amount of their purchase, your API calculates the tax and total for them instantly, without you having to manage any servers.

---

## Summary and What's Next

In this lesson, you learned how to build a simple serverless API using AWS Lambda and API Gateway. We broke down the Lambda function code, explained how to process input and return a response, and saw how the SAM template connects everything together.

Next, you will get hands-on practice by working with Lambda functions and API Gateway in the CodeSignal environment. You will write your own code, test it, and see how serverless applications work in real time. Let's get started!

## Modify Lambda Tax Calculation Rate

Now that you've learned how Lambda functions process requests and return responses, it's time to make your first code modification to see how changes affect the API output.

You'll work with the tax calculator function from the lesson and update it to use a different tax rate. Currently, the function calculates a 7% tax, but you need to change it to 8.5% instead.

Your task is to:

* Find the line where tax is calculated in the `main.py` file
* Change the tax rate from `0.07` to `0.085`
* Test how this change affects the API response

Look for the TODO comment to guide you to the exact line that needs updating. After making the change, you can see how a purchase amount of 100 now results in a tax of 8.50 and a total of 108.50 instead of the original 7.00 and 107.00.

This hands-on practice will help you understand how small code changes in Lambda functions directly impact what users receive from your API.

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    # TODO: Change the tax rate from 7% to 8.5%
    tax = round(amount * 0.07, 2)
    total = round(amount + tax, 2)
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"amount": amount, "tax": tax, "total": total})
    }
```

Here is the completed code with the tax rate updated to 8.5%:

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    tax = round(amount * 0.085, 2)
    total = round(amount + tax, 2)
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"amount": amount, "tax": tax, "total": total})
    }
```

## Fix the Broken API Response

Nice work on making your first Lambda function modification! Now it's time to tackle a debugging challenge that many developers face when working with API Gateway integration.

You have a Lambda function that builds and deploys without any errors, but when you test the API, something isn't quite right with the response format. The function processes the tax calculation correctly, but the data coming back from the API looks broken.

Your objective is to identify and fix the JSON formatting bug in the Lambda response. Run the code first to see the malformed output, then examine the handler function to find what is wrong with how the response is being returned.

Remember from the lesson that API Gateway expects the `body` field in your Lambda response to be a JSON string, not a Python dictionary. Once you spot the issue and fix it, your API will return properly formatted JSON data that clients can use correctly.

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    tax = round(amount * 0.07, 2)
    total = round(amount + tax, 2)
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": {"amount": amount, "tax": tax, "total": total}
    }
```

Here is the fixed code with the `body` converted to a JSON string using `json.dumps(...)`:

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    tax = round(amount * 0.07, 2)
    total = round(amount + tax, 2)
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"amount": amount, "tax": tax, "total": total})
    }
```

## Enhance Lambda Response with Tax Rate

Excellent work debugging that JSON response issue! Now you're ready to enhance your API by making it more informative for clients who use your function.

Currently, your Lambda function returns the amount, tax, and total, but it doesn't tell the client what tax rate was used in the calculation. This information could be valuable for users who want to understand how the tax was calculated or display it in their applications.

Your objective is to add a new field called `tax_rate` to the API response that contains the decimal value representing the tax rate used (which is currently 0.07 for 7%). Look for the TODO comment in the `main.py` file to see exactly where you need to make this change.

After completing this enhancement, when someone sends a request with an amount of 100, the response will include not just the calculated values, but also show them that a 7% tax rate was applied. This makes your API more transparent and helps clients understand exactly how their calculations were processed!

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    tax = round(amount * 0.07, 2)
    total = round(amount + tax, 2)
    # TODO: Add "tax_rate": 0.07 to the response dictionary below
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"amount": amount, "tax": tax, "total": total})
    }
```

Here is the completed code with `tax_rate` added to the response body:

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    tax = round(amount * 0.07, 2)
    total = round(amount + tax, 2)
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"amount": amount, "tax": tax, "total": total, "tax_rate": 0.07})
    }
```

## Add Input Validation to Lambda Function

Excellent work on enhancing your API with additional response data! Now it's time to make your Lambda function production-ready by adding proper error handling for invalid inputs.

Currently, your function assumes all requests will contain valid data, but real-world APIs need to handle bad inputs gracefully. Your function might crash or return incorrect results when users send missing data, negative numbers, or text instead of numbers.

Your objective is to add validation checks that protect your API from invalid requests. You need to:

* Check if the `"amount"` field exists in the request
* Verify that the amount can be converted to a valid number
* Ensure that the amount is a positive value (greater than 0)
* Return proper error responses with status code 400 when validation fails

Look for the TODO comments in the code to guide you on where to add these validation checks. When validation fails, your function should return helpful error messages that tell clients exactly what went wrong.

This exercise will teach you essential input validation patterns that make serverless APIs reliable and user-friendly in production environments.

```python
import json

def handler(event, context):
    try:
        body = json.loads(event.get("body") or "{}")
        
        # TODO: Check if "amount" field exists in the request body
        
        # TODO: Try to convert amount to float and handle conversion errors
        amount = float(body["amount"])
        
        # TODO: Check if amount is positive (greater than 0)
        
        # Calculate tax and total
        tax = round(amount * 0.07, 2)
        total = round(amount + tax, 2)
        
        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"amount": amount, "tax": tax, "total": total})
        }
        
    except json.JSONDecodeError:
        return {
            "statusCode": 400,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"error": "Invalid JSON in request body"})
        }
```

Here is the completed code with all three validation checks added:

```python
import json

def handler(event, context):
    try:
        body = json.loads(event.get("body") or "{}")
        
        if "amount" not in body:
            return {
                "statusCode": 400,
                "headers": {"Content-Type": "application/json"},
                "body": json.dumps({"error": "Missing required field: amount"})
            }
        
        try:
            amount = float(body["amount"])
        except (ValueError, TypeError):
            return {
                "statusCode": 400,
                "headers": {"Content-Type": "application/json"},
                "body": json.dumps({"error": "amount must be a valid number"})
            }
        
        if amount <= 0:
            return {
                "statusCode": 400,
                "headers": {"Content-Type": "application/json"},
                "body": json.dumps({"error": "amount must be greater than 0"})
            }
        
        # Calculate tax and total
        tax = round(amount * 0.07, 2)
        total = round(amount + tax, 2)
        
        return {
            "statusCode": 200,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"amount": amount, "tax": tax, "total": total})
        }
        
    except json.JSONDecodeError:
        return {
            "statusCode": 400,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps({"error": "Invalid JSON in request body"})
        }
```

## Convert Tax Calculator to Tip Calculator

Perfect work on building robust input validation! Now you're ready to apply your Lambda skills to a completely different business scenario by transforming your function to handle multiple input parameters.

Your task is to convert the existing tax calculator into a tip calculator that helps users calculate tips for restaurant bills. The new function should accept both an amount (the bill total) and a `tip_percentage` from the request, calculate the tip amount, and return the total, including the tip.

Here's what you need to modify:

* Extract both `"amount"` and `"tip_percentage"` from the request body.
* Set a default tip percentage of 15% (0.15) when no tip percentage is provided.
* Change the calculation logic from tax to tip calculation.
* Update the response field names to `"tip"` and `"total_with_tip"`.

Look for the TODO comments to guide you through each change. When complete, your API will handle requests like `{"amount": 50, "tip_percentage": 0.20}` and return the tip amount plus the total with the tip included.

This exercise will strengthen your understanding of multi-parameter Lambda functions and show you how to adapt serverless APIs for different business requirements!

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    # TODO: Extract tip_percentage from the request body with a default of 0.15 (15%)
    tax = round(amount * 0.07, 2)
    # TODO: Calculate tip instead of tax using amount * tip_percentage
    total = round(amount + tax, 2)
    # TODO: Calculate total_with_tip instead of total
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        # TODO: Update the response fields to include tip_percentage, tip, and total_with_tip
        "body": json.dumps({"amount": amount, "tax": tax, "total": total})
    }
```

Here is the completed code converted into a tip calculator:

```python
import json

def handler(event, context):
    body = json.loads(event.get("body") or "{}")
    amount = float(body.get("amount", 0))
    tip_percentage = float(body.get("tip_percentage", 0.15))
    tip = round(amount * tip_percentage, 2)
    total_with_tip = round(amount + tip, 2)
    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({
            "amount": amount,
            "tip_percentage": tip_percentage,
            "tip": tip,
            "total_with_tip": total_with_tip
        })
    }
```